In [91]:
import numpy as np
import astropy.units as u
from astropy import constants as const

def solid_angle(r):
    """
    Calculates the solid angle of the source.

    Parameters:
    - r: Radius of the source [deg]

    Returns:
    - Calculated solid angle of the source [sr].
    """
    r_rad = (r * u.deg).to(u.rad).value
    return (2 * np.pi * (1 - np.cos(r_rad))) * u.sr

In [92]:
from astropy.io import fits
from astropy.wcs import WCS

def extract_fits_params(fits_path):
    """
    Extracts observational parameters from a FITS spectral cube.

    Reads beam dimensions, pixel size, and integrated flux directly
    from the file, so they do not need to be entered manually.

    Parameters:
    - fits_path: Path to the FITS cube [string]

    Returns a dict with:
    - dx, dy        : Pixel size [arcsec]
    - th_maj, th_min: Beam FWHM [arcsec]
    - int_flux      : Integrated flux ∫I_ν dV [Jy * km/s]
    """
    with fits.open(fits_path) as hdul:
        hdr  = hdul[0].header
        data = hdul[0].data

    dx     = abs(hdr['CDELT1']) * 3600  # deg -> arcsec
    dy     = abs(hdr['CDELT2']) * 3600
    th_maj = hdr['BMAJ'] * 3600
    th_min = hdr['BMIN'] * 3600

    cdelt3 = hdr['CDELT3']
    ctype3 = hdr.get('CTYPE3', '').upper()
    cunit3 = hdr.get('CUNIT3', '').upper()

    if 'FREQ' in ctype3:
        restfreq = hdr.get('RESTFRQ', hdr.get('RESTFREQ', None))
        if restfreq is None:
            raise ValueError("Rest frequency (RESTFRQ) not found in header.")
        dv_kms = abs(cdelt3 / restfreq) * const.c.to(u.km/u.s).value
    else:
        dv_kms = abs(cdelt3) if 'KM/S' in cunit3 else abs(cdelt3) / 1e3  # m/s -> km/s

    if data.ndim == 4:
        data = data[0]

    int_flux = np.nansum(data) * dv_kms  # Jy/beam * km/s, beam correction applied in mol_upper_state_cd

    return {
        'dx':       dx,
        'dy':       dy,
        'th_maj':   th_maj,
        'th_min':   th_min,
        'int_flux': int_flux,
    }

In [93]:
def integrated_flux_from_tsv(tsv_path, rest_freq_ghz, threshold=None, baseline_regions=None):
    """
    Compute integrated flux ∫I_nu dV from a CARTA spectral profile TSV.

    Parameters
    ----------
    tsv_path : str
        Path to TSV file.
    rest_freq_ghz : float
        Rest frequency [GHz].
    threshold : float, optional
        Threshold in Jy/beam for defining the line region.
        If None, uses 0 after baseline subtraction.
    baseline_regions : list of tuple, optional
        Frequency regions [(f1, f2), (f3, f4), ...] in GHz used to estimate baseline.

    Returns
    -------
    float
        Integrated flux [Jy/beam km/s]
    """
    data = np.loadtxt(tsv_path, comments="#")
    freq_ghz = data[:, 0]
    intensity = data[:, 1]

    # Estimate and subtract baseline
    baseline = 0.0
    if baseline_regions is not None:
        mask = np.zeros_like(freq_ghz, dtype=bool)
        for lo, hi in baseline_regions:
            mask |= (freq_ghz >= lo) & (freq_ghz <= hi)
        if np.any(mask):
            baseline = np.nanmean(intensity[mask])

    intensity_sub = intensity - baseline

    velocity_kms = (rest_freq_ghz - freq_ghz) / rest_freq_ghz * const.c.to(u.km/u.s).value

    if threshold is None:
        threshold = 0.0

    peak_idx = np.nanargmax(intensity_sub)
    start = peak_idx
    end = peak_idx

    while start > 0 and intensity_sub[start - 1] > threshold:
        start -= 1
    while end < len(intensity_sub) - 1 and intensity_sub[end + 1] > threshold:
        end += 1

    return abs(np.trapezoid(intensity_sub[start:end+1], velocity_kms[start:end+1]))

In [94]:
def mol_upper_state_cd(A_ul, int_flux, dx, dy, th_maj, th_min, r):
    """
    Calculates the molecular upper state column density.

    N_u = (4π / (A_ul * Ω * h * c)) * ∫I_ν dV * (dx*dy * 4ln2) / (π * θ_maj * θ_min)

    Parameters:
    - A_ul:      Einstein A coefficient [s^-1]
    - int_flux: Integrated spectral intensity from the extracted spectrum [Jy/beam * km/s]
    - dx, dy:    Angular pixel/cell size [arcsec]
    - th_maj:    Beam major axis FWHM [arcsec]
    - th_min:    Beam minor axis FWHM [arcsec]
    - r:         Radius of the source [arcsec]

    Returns:
    - N_u:       Upper state column density [cm^-2]
    """
    omega = solid_angle(r / 3600).value  # arcsec -> deg
    arcsec_to_rad = (1 * u.arcsec).to(u.rad).value
    omega_pixel = dx * dy * arcsec_to_rad**2
    omega_beam = np.pi * th_maj * th_min * arcsec_to_rad**2 / (4 * np.log(2))
    beam_corr = omega_pixel / omega_beam
    int_flux_cgs = int_flux * 1e-23 * 1e5
    h = const.h.cgs.value   # erg*s
    c = const.c.cgs.value   # cm/s
    N_u = (4 * np.pi * int_flux_cgs * beam_corr) / (A_ul * omega * h * c)

    return N_u  # cm^-2

In [95]:
from scipy.interpolate import interp1d

def partition_func(T_rot, temps, Q_vals):
    """
    Interpolates the partition function Q(T_rot) from a list of values.
    Table data can be obtained from SPLATALOGUE for a specific molecule.

    Parameters:
    - T_rot:   Rotational temperature [K]
    - temps:   Array of temperatures at which Q is tabulated [K]
    - Q_vals:  Corresponding partition function values

    Returns:
    - Q:       Interpolated partition function at T_rot
    """
    log_interp = interp1d(np.log(temps), np.log(Q_vals), kind='linear', fill_value='extrapolate')
    return float(np.exp(log_interp(np.log(T_rot))))


def upper_state_degeneracy(J, g_nuclear=1):
    """
    Upper-state degeneracy for a linear rotor:
    g_u = (2J + 1) * g_nuclear
    """
    return (2 * J + 1) * g_nuclear

In [96]:
def total_mol_cd(N_u, g_u, Q_rot, E_u, T_rot):
    """
    Calculates the total molecular column density.

    N_T = N_u * Q(T_rot) * e^(E_u / T_rot) / g_u

    Parameters:
    - N_u:   Upper state column density [cm^-2], from mol_upper_state_cd()
    - g_u:   Upper state degeneracy
    - Q_rot: Partition function at T_rot
    - E_u:   Upper state energy [K]  (E_u/k_B, already in temperature units)
    - T_rot: Rotational temperature [K]

    Returns:
    - N_T:   Total molecular column density [cm^-2]
    """
    N_T = (N_u * Q_rot * np.exp(E_u / T_rot)) / g_u
    return N_T  # cm^-2

In [97]:
# --- Extract beam/pixel params from FITS ---
params = extract_fits_params('CS-97dot9-dot98.FITS')

# --- Integrated flux from CARTA spectral profile ---
rest_freq_ghz = 97.9809702  # CS J=2->1 rest frequency [GHz]
int_flux = integrated_flux_from_tsv('csdetectionghzsum.tsv', rest_freq_ghz)
print(f"Integrated flux: {int_flux}")

# --- Molecular parameters (fill in from SPLATALOGUE/CDMS) ---
A_ul   = 1.69E-05   # Einstein A coefficient [s^-1]
E_u    = 7.054      # Upper state energy [K]
J      = 2          # Upper rotational quantum number

temps  = [9.375, 18.75, 37.5, 75.0, 150.0, 225.0, 300.0]
Q_vals = [8.316, 16.285, 32.240, 64.150, 127.968, 191.823, 255.505]
temps, Q_vals = zip(*sorted(zip(temps, Q_vals)))

# --- Physical inputs ---
T_rot  = 24.4   # Rotational temperature [K]
r      = 1.5    # arcsec

# --- Pipeline ---
g_u   = upper_state_degeneracy(J)
Q_rot = partition_func(T_rot, temps, Q_vals)

N_u = mol_upper_state_cd(
    A_ul     = A_ul,
    int_flux = int_flux,
    dx       = params['dx'],
    dy       = params['dy'],
    th_maj   = params['th_maj'],
    th_min   = params['th_min'],
    r        = r,
)

N_T = total_mol_cd(N_u, g_u, Q_rot, E_u, T_rot)

print(f"N_u = {N_u:.3e} cm^-2")
print(f"N_T = {N_T:.3e} cm^-2")

Integrated flux: 15.521034550515466
N_u = 6.184e+12 cm^-2
N_T = 3.486e+13 cm^-2


In [98]:
# just for verifying the validity of file inputs
params['dx'], params['dy'], params['th_maj'], params['th_min']

(0.11999999993162398,
 0.11999999993162398,
 1.02076637744916,
 0.7040538191793599)